# Klasik Görüntü İşleme ile Mikroskobik Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

**Ders:** Görüntü İşleme — Yüksek Lisans Ara Sınav Projesi
**Veri seti:** Cell Tracking Challenge — Fluo-N2DH-GOWT1 (eğitim arşivi)
**Random seed:** `31084` (öğrenci numarası son 5 hane `29058` + `2026`)

Bu notebook baştan sona sırayla çalıştırıldığında aynı veri bölünmesini, aynı deneyleri ve aynı sayısal sonuçları üretir.

> GitHub'a yüklenen sürümde veri setine ait görüntü çıktıları temizlenmiştir (ödev Bölüm 18.5).

## 0. Kurulum ve konfigürasyon

In [ ]:
# Colab'da çalışıyorsak eksik paketleri kur
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagecodecs', 'tifffile'], check=True)

In [ ]:
import os, json, time, zipfile, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tifffile
import skimage
from scipy import ndimage as ndi

# ---- Sabitler ----
STUDENT_ID_LAST5 = 29058
SEED = STUDENT_ID_LAST5 + 2026          # = 31084
DATASET_URL = 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip'

# ---- Klasörler ----
PROJECT_DIR = Path.cwd()
DATA_DIR    = PROJECT_DIR / 'data'                      # .gitignore'da: veri GitHub'a gitmez
DATASET_DIR = DATA_DIR / 'Fluo-N2DH-GOWT1'
FIG_DIR     = PROJECT_DIR / 'figures'                   # veri görüntüsü içermeyen grafikler
PRIVATE_FIG_DIR = PROJECT_DIR / 'figures_private'       # veri görüntüsü içeren şekiller (sadece rapor)
for d in (DATA_DIR, FIG_DIR, PRIVATE_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('SEED =', SEED)
print('OpenCV', cv2.__version__, '| scikit-image', skimage.__version__, '| NumPy', np.__version__)

In [ ]:
# Veri seti yoksa resmi bağlantıdan indir ve aç
if not DATASET_DIR.exists():
    zip_path = DATA_DIR / 'Fluo-N2DH-GOWT1.zip'
    if not zip_path.exists():
        print('İndiriliyor...')
        urllib.request.urlretrieve(DATASET_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
print(sorted(p.name for p in DATASET_DIR.iterdir()))

In [ ]:
def frame_path(seq: str, t: int) -> Path:
    """Ham görüntü yolu, ör. ('01', 5) -> .../01/t005.tif"""
    return DATASET_DIR / seq / f't{t:03d}.tif'

def read_frame(seq: str, t: int) -> np.ndarray:
    return tifffile.imread(frame_path(seq, t))

def n_frames(seq: str) -> int:
    return len(list((DATASET_DIR / seq).glob('t*.tif')))

## 1. Veri setinin ön incelemesi (Ödev Bölüm 2)

Bu bölümde **sadece ham görüntüler** (her iki sekans) ve **Sequence 01'in** referans maskeleri incelenir.
Sequence 02'nin referans anotasyonlarına (`02_GT`) test kilidine kadar dokunulmaz.

### 1.1 Boyut, veri tipi ve kare sayısı

In [ ]:
rows = []
for seq in ['01', '02']:
    img = read_frame(seq, 0)
    rows.append({'Sekans': seq, 'Kare sayısı': n_frames(seq), 'Boyut': img.shape,
                 'dtype': str(img.dtype), 'Teorik aralık': f'0–{np.iinfo(img.dtype).max}'})
pd.DataFrame(rows)

### 1.2 Kare bazında yoğunluk istatistikleri

In [ ]:
stats = []
for seq in ['01', '02']:
    for t in range(n_frames(seq)):
        im = read_frame(seq, t).astype(np.float64)
        stats.append({'seq': seq, 't': t, 'mean': im.mean(), 'std': im.std(),
                      'min': im.min(), 'max': im.max(),
                      'p01': np.percentile(im, 1), 'p50': np.percentile(im, 50),
                      'p99': np.percentile(im, 99),
                      'frac_zero': (im == 0).mean()})
stats = pd.DataFrame(stats)
summary = stats.groupby('seq')[['mean', 'std', 'min', 'max', 'p01', 'p50', 'p99', 'frac_zero']].agg(['mean', 'min', 'max']).round(2)
summary.T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for seq, c in [('01', 'tab:blue'), ('02', 'tab:orange')]:
    s = stats[stats.seq == seq]
    axes[0].plot(s.t, s['mean'], c=c, label=f'Seq {seq}')
    axes[1].plot(s.t, s['p99'], c=c, label=f'Seq {seq}')
    axes[2].plot(s.t, s['max'], c=c, label=f'Seq {seq}')
for ax, title in zip(axes, ['Ortalama yoğunluk', '99. yüzdelik', 'Maksimum']):
    ax.set_title(title); ax.set_xlabel('Kare (t)'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig_intensity_over_time.png', dpi=150); plt.show()

### 1.3 Örnek kareler ve histogramlar

In [ ]:
sample_t = [0, 45, 91]
fig, axes = plt.subplots(2, 4, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 1, 1, 1.3]})
for r, seq in enumerate(['01', '02']):
    for c, t in enumerate(sample_t):
        im = read_frame(seq, t)
        axes[r, c].imshow(im, cmap='gray', vmin=0, vmax=255)
        axes[r, c].set_title(f'Seq {seq} — t{t:03d} (0–255 sabit ölçek)'); axes[r, c].axis('off')
        axes[r, 3].hist(im.ravel(), bins=256, range=(0, 256), histtype='step', log=True, label=f't{t:03d}')
    axes[r, 3].set_title(f'Seq {seq} histogram (log y)'); axes[r, 3].set_xlabel('Yoğunluk'); axes[r, 3].legend()
plt.tight_layout(); plt.savefig(PRIVATE_FIG_DIR / 'fig_samples_hist.png', dpi=120); plt.show()

### 1.4 Hücre boyutu ve temas eden hücreler (yalnızca Sequence 01 referans maskeleri)

`01_GT/SEG` klasöründe 92 karenin yalnızca 30'u için piksel düzeyinde maske vardır ve bu maskelerin çoğu **kısmidir**
(karedeki hücrelerin yalnızca bir kısmı işaretlenmiştir). Önce her SEG maskesindeki hücre sayısı aynı karenin TRA sayısıyla karşılaştırılır;
alan ve temas analizi yalnızca **tam işaretli** karelerde yapılır. Bu bilgi ileride minimum alan eşiği ve yapı elemanı boyutu için fiziksel referans olacaktır.

In [ ]:
def tra_count(seq: str, t: int) -> int:
    """Referans hücre sayısı: TRA etiket görüntüsündeki farklı pozitif kimlik sayısı (Ödev Bölüm 4)."""
    lab = tifffile.imread(DATASET_DIR / f'{seq}_GT' / 'TRA' / f'man_track{t:03d}.tif')
    return int(np.count_nonzero(np.unique(lab)))

seg_files = sorted((DATASET_DIR / '01_GT' / 'SEG').glob('man_seg*.tif'))
cov = []
for f in seg_files:
    t = int(f.stem.replace('man_seg', ''))
    n_seg = int(np.count_nonzero(np.unique(tifffile.imread(f))))
    cov.append({'t': t, 'SEG hücre': n_seg, 'TRA hücre': tra_count('01', t)})
cov = pd.DataFrame(cov)
cov['tam_işaretli'] = cov['SEG hücre'] == cov['TRA hücre']
full_t = cov.loc[cov.tam_işaretli, 't'].tolist()
print(f'SEG maskeli kare: {len(cov)} | tam işaretli: {len(full_t)} -> {full_t}')
cov.T

In [ ]:
areas, touching_rows = [], []
k = np.ones((3, 3), np.uint8)
for t in full_t:
    f = DATASET_DIR / '01_GT' / 'SEG' / f'man_seg{t:03d}.tif'
    lab = tifffile.imread(f)
    ids = [i for i in np.unique(lab) if i != 0]
    n_touch = 0
    for i in ids:
        m = (lab == i)
        areas.append(m.sum())
        ring = cv2.dilate(m.astype(np.uint8), k, iterations=2).astype(bool) & ~m
        if np.any((lab[ring] != 0) & (lab[ring] != i)):
            n_touch += 1
    touching_rows.append({'file': f.name, 'n_cells_seg': len(ids), 'n_touching': n_touch})
areas = np.array(areas)
touch = pd.DataFrame(touching_rows)
eq_diam = 2 * np.sqrt(areas / np.pi)
print(f'Analiz edilen tam işaretli kare: {len(full_t)}, hücre: {len(areas)}')
print(f'Hücre alanı (piksel): medyan={np.median(areas):.0f}, 5.yüzdelik={np.percentile(areas,5):.0f}, 95.yüzdelik={np.percentile(areas,95):.0f}, min={areas.min()}, max={areas.max()}')
print(f'Eşdeğer çap (piksel): medyan={np.median(eq_diam):.1f}, aralık={eq_diam.min():.1f}–{eq_diam.max():.1f}')
print(f'Temas eden (≤2 piksel komşu) hücre oranı: {touch.n_touching.sum() / touch.n_cells_seg.sum():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(areas, bins=40, color='tab:blue'); axes[0].set_title('Seq 01 hücre alanı dağılımı'); axes[0].set_xlabel('Alan (piksel)')
axes[1].hist(eq_diam, bins=40, color='tab:green'); axes[1].set_title('Eşdeğer çap'); axes[1].set_xlabel('Çap (piksel)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig_cell_size_seq01.png', dpi=150); plt.show()

### 1.5 Düzensiz arka plan ve hücreler arası parlaklık farkı

In [ ]:
t_ex = full_t[0]
im = read_frame('01', t_ex)
lab = tifffile.imread(DATASET_DIR / '01_GT' / 'SEG' / f'man_seg{t_ex:03d}.tif')
# Arka planı ölçmek için hücreleri (genişletilmiş referans maskesi) dışarıda bırak, 8x8 ızgarada ortalama al
bg_mask = cv2.dilate((lab > 0).astype(np.uint8), np.ones((15, 15), np.uint8)) == 0
G = 8; h, w = im.shape; bg_grid = np.zeros((G, G))
for i in range(G):
    for j in range(G):
        blk = im[i*h//G:(i+1)*h//G, j*w//G:(j+1)*w//G]; m = bg_mask[i*h//G:(i+1)*h//G, j*w//G:(j+1)*w//G]
        bg_grid[i, j] = blk[m].mean() if m.any() else np.nan
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(im, cmap='gray'); axes[0].set_title(f'Seq 01 t{t_ex:03d} (otomatik kontrast)'); axes[0].axis('off')
hm = axes[1].imshow(bg_grid, cmap='magma'); plt.colorbar(hm, ax=axes[1], fraction=.046)
axes[1].set_title('Hücre dışı arka plan ortalaması (8x8 ızgara)'); axes[1].axis('off')
print(f'Arka plan ızgara ortalaması: min={np.nanmin(bg_grid):.2f}, max={np.nanmax(bg_grid):.2f}')
if lab is not None:
    means = [im[lab == i].mean() for i in np.unique(lab) if i != 0]
    axes[2].hist(means, bins=15, color='tab:purple'); axes[2].set_title(f't{t_ex:03d}: hücre başına ortalama parlaklık'); axes[2].set_xlabel('Ortalama yoğunluk')
plt.tight_layout(); plt.savefig(PRIVATE_FIG_DIR / 'fig_background_brightness.png', dpi=120); plt.show()
if lab is not None:
    print(f'Hücre ortalama parlaklıkları: min={min(means):.1f}, max={max(means):.1f}  (oran ≈ {max(means)/max(min(means),1):.1f}x)')